# Lab 6

Lo primero que vamos a hacer es cargar los datos, con esto convertimos los json a columnas y luego los cargamos, una cosa los json hijos tendran esta estructura `padre_hijo`

In [32]:
import re
import pandas as pd
import nltk
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from textblob import TextBlob

In [47]:
import json
import pandas as pd

def load_json_lines_to_df(filepath, encoding="utf-16"):
    all_objs = []

    with open(filepath, "r", encoding=encoding) as f:
        for line_number, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                # Convertir el JSON de la línea a diccionario
                obj = json.loads(line)
                all_objs.append(obj)
            except json.JSONDecodeError as e:
                print(f"⚠️ Error JSON en línea {line_number}: {e}")
                # Opcional: guardar la línea completa como string para revisarla
                all_objs.append({"raw_line": line})

    # Normalizar JSON, aplastando diccionarios anidados
    df = pd.json_normalize(all_objs, sep='_', errors='ignore')

    return df

# Cargar dataset
df_arevalo = load_json_lines_to_df("Data/tioberny.txt")



cols_to_show = ['id', 'rawContent', 'user_username', 'user_displayname']
print(df_arevalo[[c for c in cols_to_show if c in df_arevalo.columns]].head())


                    id                                         rawContent  \
0  1834281080029110288  _\nConfirmado Compañeres,\n\nEl impuesto por l...   
1  1834252464092069901  #URGENTE Lo que los medios #faferos no informa...   
2  1834280919336976681  @IvanDuque @BArevalodeLeon Con que usaste PEGA...   
3  1834280512933732694  @IvanDuque @BArevalodeLeon Entre Ellos se enti...   
4  1834279986254987428  El presidente @BArevalodeLeon y la vicepreside...   

    user_username        user_displayname  
0  La_ReVoluZzion      The_ReVoluZZzioN 🫡  
1      XelaNewsGt                XelaNews  
2       M24095273  VIVAlafuenteDEtodaVIDA  
3    carlosalbesc  Carlos Alberto Escobar  
4      Brenda_AGN             Brenda Lari  


In [48]:
df_arevalo.columns

Index(['id', 'id_str', 'url', 'date', 'lang', 'rawContent', 'replyCount',
       'retweetCount', 'likeCount', 'quoteCount',
       ...
       'quotedTweet_quotedTweet_inReplyToTweetIdStr',
       'quotedTweet_quotedTweet_inReplyToUser',
       'quotedTweet_quotedTweet_source', 'quotedTweet_quotedTweet_sourceUrl',
       'quotedTweet_quotedTweet_sourceLabel',
       'quotedTweet_quotedTweet_media_photos',
       'quotedTweet_quotedTweet_media_videos',
       'quotedTweet_quotedTweet_media_animated',
       'quotedTweet_quotedTweet_card', 'quotedTweet_quotedTweet__type'],
      dtype='object', length=203)

## Limpieza

Lo que vamos a hacer es limpiar los datos. haciendo pipeline de limpieza que usamos en nuestros labs anteriores

In [49]:
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS  # Opcional si quieres combinar

# Descargar stopwords en español de NLTK
nltk.download('stopwords')

# Stopwords completas en español de NLTK
stop_words = set(stopwords.words('spanish'))

# Si quieres agregar palabras extra comunes en español
extra_words = {
    "como", "para", "con", "sin", "sobre", "entre", "hasta", "desde",
    "que", "quien", "quienes", "cual", "cuales", "cuando", "donde",
    "por", "el", "la", "los", "las", "un", "una", "unos", "unas",
    "y", "o", "pero", "si", "aunque", "porque"
}

# Actualizamos stop_words
stop_words.update(extra_words)

# Números permitidos en tweets
allowed_numbers = {"1945", "911", "2008", "2014", "1980", "2013", "2016", "2011"}



[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mathew\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [50]:
def convert_to_lower(word): #Convierte a minúsculas
    return word.lower()

def remove_special_characters(word): #Remueve comillas. arrobas y hashtagas
    return word.replace("#", "").replace("@", "").replace("'", "")

def remove_url(word): #Busca urls y las remueve
    text = re.sub(r'http\S+|www\S+|https\S+', '', word)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_emojis(word): #Busca emojis y los borra
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F" 
        "\U0001F300-\U0001F5FF" 
        "\U0001F680-\U0001F6FF" 
        "\U0001F1E0-\U0001F1FF"  
        "\U00002700-\U000027BF"  
        "\U0001F900-\U0001F9FF"  
        "\U00002600-\U000026FF"  
        "\U00002B00-\U00002BFF" 
        "\U0001FA70-\U0001FAFF" 
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', word).strip()

# Función para limpiar el texto
def clean_text(text):
    # Quitar puntuación
    text = re.sub(r'[^\w\s]', '', str(text))
    # Tokenizar y eliminar stopwords
    tokens = [word for word in text.split() if word.lower() not in stop_words]
    return " ".join(tokens)


def extract_numbers(text):
    return re.findall(r'\d+', str(text)) #busca y extrae números en el dataset

def count_words(dataset, column): #Esta función cuenta las palabras en cada tweet
    all_words = ''
    for i in dataset[column]:
        all_words = all_words + i + ' '

    all_words = all_words.split()

    frequency = Counter(all_words)

    print("{:<10} {:<10}".format("Palabra", "Frecuencia"))
    print("-" * 20)
    for palabra, freq in frequency.most_common():
        print("{:<10} {:<10}".format(palabra, freq))

def remove_unwanted_numbers(text): #Remueve números que no son permitidos en los tweets
    return re.sub(r'\b(?!' + '|'.join(allowed_numbers) + r')\d+\b', '', str(text))

def get_words(text_series): #Obtener y separar palabras por tweet
    words = []
    for text in text_series:
        words.extend(str(text).split())  
    return words


In [51]:
df_arevalo['rawContent'] = df_arevalo['rawContent'].astype(str).apply(convert_to_lower)
df_arevalo['rawContent'] = df_arevalo['rawContent'].astype(str).apply(remove_special_characters)
df_arevalo['rawContent'] = df_arevalo['rawContent'].astype(str).apply(remove_url)
df_arevalo['rawContent'] = df_arevalo['rawContent'].astype(str).apply(remove_emojis)


df_arevalo["rawContent"] = df_arevalo["rawContent"].apply(clean_text)


df_arevalo["rawContent"] = df_arevalo["rawContent"].apply(remove_unwanted_numbers)

df_arevalo["rawContent"].head()


0    _ confirmado compañeres impuesto usembassyguat...
1    urgente medios faferos informaron ayer acerca ...
2    ivanduque barevalodeleon usaste pegasus espiar...
3    ivanduque barevalodeleon entienden bien cuadra...
4    presidente barevalodeleon vicepresidenta karin...
Name: rawContent, dtype: object